# Variance ceiling and predictor saturation, parameterized by variant class and tool set

Same framework as [`variance_ceiling_master_file.ipynb`](variance_ceiling_master_file.ipynb),
but with the multiplicative SE calibration removed. Noise is estimated **directly** as the
empirical variance of AC=1 synonymous variants on the unassociated (null-trait) panel, pooled
by trait. There is no calibration factor `c` and no per-variant standard error in the noise
model.

`variant_class` comes from `config_variant_classes.yaml` (via
`utils/variant_filtering.load_variant_class`/`scan_variants`) and the tool set comes from
`config_correlations.yaml` (via `load_config`/`pick_annos`).

There is a single, unsplit ceiling here -- no within/outside contrast (e.g. TED domain); for
that see
[`variance_ceiling_ted_contrast_master_file_trait_noise.ipynb`](variance_ceiling_ted_contrast_master_file_trait_noise.ipynb).

Reads `MASTER_PATH` for the analysis population, plus `UNASSOC_PATH` used **only** to estimate
the per-trait noise level (never as signal).


## 1. Model and noise definition

For variant $v$ in gene-trait unit $j=(g,t)$,

$$\hat\beta_{v,j} = X_{v,j} + E_{v,j}.$$

Noise is estimated empirically, per trait, from **AC=1 synonymous variants on the unassociated
(null-trait) panel** -- synonymous-class Genebass associations for traits unrelated to the
phenotype under study, across all genes:

$$\sigma_t^2=\operatorname{Var}\big(\hat\beta_{\mathrm{AC}=1,\,\mathrm{syn}}\mid \mathrm{trait}=t\big),
\qquad \sigma_v^2=\sigma_{t(v)}^2 .$$

These variants carry no real signal, so their observed spread *is* the sampling noise. No
calibration factor and no additive offset are needed: the quantity is measured on the same
scale, at the same allele count, as the analysis variants.

Thus $\operatorname{Var}(E_{v,j}) \approx \sigma_t^2$. The main analysis is restricted to
**AC=1** (singleton) variants of the chosen `variant_class`, and every predictor in the chosen
tool set is compared on exactly the same variants (complete-case).


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import polars as pl
from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import env_override, fetch_hf_data
CONFIG_DIR   = str(REPO_ROOT / 'configs')
MASTER_PATH  = env_override('MASTER_PATH', fetch_hf_data('genebass_annotated.parquet', REPO_ROOT))
UNASSOC_PATH = env_override('UNASSOC_PATH', fetch_hf_data('genebass_unassociated.parquet', REPO_ROOT))
FIG_DIR      = Path(env_override('FIG_DIR', '../../../paper_figures'))
FIG_DIR.mkdir(exist_ok=True)

In [ ]:
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import load_config, load_variant_class, scan_variants, pick_annos, env_override

_THEME = theme_minimal() + theme(
    axis_text=element_text(size=11, lineheight=1.4),
    axis_title=element_text(size=12),
    legend_text=element_text(size=12),
    legend_title=element_text(size=12),
    plot_background=element_rect(fill='white', color='white'),
)

MASTER_SCHEMA = set(pl.scan_parquet(MASTER_PATH).collect_schema().names())
print(f'master table: {len(MASTER_SCHEMA)} cols')

## 2. Parameters

In [ ]:
# --- variant class + tool set (drive everything below; env_override(NAME, default)) ---
variant_class       = env_override('VARIANT_CLASS', 'indel')   # any key in config_variant_classes.yaml
config_file         = env_override('CONFIG_FILE', 'config_correlations.yaml')
selected_categories = env_override('NOISE_CEILING_CATEGORIES', None, 'list')   # None -> this variant class's own `tool_categories`
only_snps           = env_override('ONLY_SNPS', False, bool)

# --- gene-trait unit settings ---
MIN_N_PAIR = env_override('MIN_VARIANTS', 10, int)

N_BOOT    = env_override('N_BOOT', 2000, int)
BOOT_SEED = env_override('BOOT_SEED', 1, int)


## 3. Load config, variant class and tool set

In [ ]:
anno_cfg, all_annos = load_config(CONFIG_DIR, config_file)
vc = load_variant_class(CONFIG_DIR, variant_class)
lf = scan_variants(MASTER_PATH, vc, only_snps=only_snps)

if selected_categories is None:
    selected_categories = vc['tool_categories']

annos = pick_annos(anno_cfg, all_annos, selected_categories, MASTER_SCHEMA)
anno_to_label = dict(anno_cfg.select(['annotation', 'label']).unique().iter_rows())

print(f'variant_class = {variant_class}')
print(f'categories    = {selected_categories}')
print(f'{len(annos)} tools: {annos}')

## 4. Per-trait noise from AC=1 synonymous variants on the unassociated panel

The null-trait panel carries no annotation columns and is **not** restricted to AC=1: only ~43%
of its rows are singletons. Since `mean_pheno_value` is the *mean* phenotype over the AC
carriers, its variance falls as $\approx\sigma^2/\mathrm{AC}$, so pooling over all AC would
badly under-estimate singleton noise. We therefore join `AC` and
`consequence_synonymous_variant` in from the master table by `id` and keep only AC=1 synonymous
rows before taking the per-trait variance.

This step doesn't depend on `variant_class`/`annos` at all -- the same per-trait noise is reused
regardless of what's analysed below.


In [ ]:
COL_ID, COL_GENE, COL_TRAIT = 'id', 'gene_name', 'phenotype'
COL_MAC, COL_BETA, COL_SE = 'AC', 'mean_pheno_value', 'SE'
COL_SYNONYMOUS = 'consequence_synonymous_variant'

# The null panel has no AC / consequence columns -- bring them in from the master table by `id`.
annot = (
    pl.scan_parquet(MASTER_PATH)
    .select([COL_ID, COL_MAC, COL_SYNONYMOUS])
    .unique(subset=[COL_ID])
    .collect()
)

unassoc_df = (
    pl.scan_parquet(UNASSOC_PATH)
    .select(pl.col(COL_ID), pl.col(COL_TRAIT), pl.col(COL_BETA).alias('beta'), pl.col(COL_SE))
    .filter(pl.col('beta').is_not_null(), pl.col('beta').is_finite())
    .collect()
    .join(annot, on=COL_ID, how='inner')
    .filter(pl.col(COL_MAC) == 1, pl.col(COL_SYNONYMOUS) == 1)
)

# Per-trait noise: empirical variance of AC=1 synonymous betas, pooled by trait only.
trait_noise = (
    unassoc_df
    .group_by(COL_TRAIT)
    .agg(pl.col('beta').var(ddof=1).alias('sigma2_t'), pl.len().alias('k_t'))
)

print(f'AC=1 synonymous null rows: {unassoc_df.height:,}')
print(f'Traits: {trait_noise.height}   min variants/trait: {trait_noise["k_t"].min():,}')
s2 = trait_noise['sigma2_t']
print(f'sigma2_t  median={s2.median():.4f}  min={s2.min():.4f}  max={s2.max():.4f}')

if not np.isfinite(s2.to_numpy()).all() or s2.min() <= 0:
    raise ValueError('Non-finite or non-positive per-trait noise variance.')


### Orthogonal check: are the Genebass SEs honest?

The empirical AC=1 synonymous noise $\sigma_t^2$ and the *reported* $\overline{SE^2}$ for the
same rows are two independent estimates of the same quantity. Their ratio
$c_t=\sigma_t^2/\overline{SE^2}$ is exactly the old calibration factor, now computed per trait.
$c_t\approx1$ means the reported SEs are already well calibrated, so dropping the calibration
step costs nothing. This is diagnostic only -- nothing downstream consumes it.


In [ ]:
se_check = (
    unassoc_df
    .filter(pl.col(COL_SE).is_not_null(), pl.col(COL_SE).is_finite())
    .group_by(COL_TRAIT)
    .agg(
        pl.col('beta').var(ddof=1).alias('sigma2_t'),
        (pl.col(COL_SE) ** 2).mean().alias('mean_se2'),
    )
    .with_columns((pl.col('sigma2_t') / pl.col('mean_se2')).alias('c_t'))
)

ct = se_check['c_t']
print(f'c_t across {se_check.height} traits: median={ct.median():.4f}  '
      f'q10={ct.quantile(0.1):.4f}  q90={ct.quantile(0.9):.4f}')
print('c_t == 1 would mean the reported Genebass SE is exactly right.')


## 5. Load complete-case singleton variants for the chosen class + tools

A variant enters the analysis only if it is AC=1, matches `variant_class`, and every tool in
`annos` has a finite score. For each variant we define $\sigma_v^2 = \sigma_{t(v)}^2$ (its
trait's empirical noise variance).


In [ ]:
predictor_exprs = [pl.col(a).cast(pl.Float64) for a in annos]

analysis_variants = (
    lf
    .filter(
        pl.col(COL_MAC) == 1,
        pl.col(COL_BETA).is_not_null(), pl.col(COL_BETA).is_finite(),
        *[pl.col(a).is_not_null() & pl.col(a).is_finite() for a in annos],
    )
    .select(
        pl.col(COL_ID), pl.col(COL_GENE), pl.col(COL_TRAIT),
        pl.col(COL_BETA).cast(pl.Float64).alias('beta'),
        *predictor_exprs,
    )
    .collect()
    .join(trait_noise.select([COL_TRAIT, 'sigma2_t']), on=COL_TRAIT, how='inner')
    .with_columns(pl.col('sigma2_t').alias('sigma2'))
)

print(f'Complete-case {variant_class} singleton rows: {analysis_variants.height:,}')
print(f'Minimum trait noise sigma2: {analysis_variants["sigma2"].min():.6f}')

if analysis_variants.height == 0:
    raise ValueError(
        f'No complete-case singleton variants for variant_class={variant_class!r} with '
        f'selected_categories={selected_categories!r} (tools: {annos}). This usually means the '
        "tool set includes a predictor that's always null for this variant class (e.g. a "
        "'missense' category tool on non-missense variants) -- check vc['tool_categories'] for "
        'the categories this variant class actually has scores for.'
    )
if analysis_variants['sigma2'].min() <= 0:
    raise ValueError('Trait noise variance is non-positive for at least one variant.')


## 6. Fix the gene–trait units once

A gene–trait unit is retained only if it has at least `MIN_N_PAIR` complete-case variants. This
set is then fixed for the rest of the notebook.

In [ ]:
common_pairs = (
    analysis_variants.group_by(COL_GENE, COL_TRAIT).agg(pl.len().alias('n'))
    .filter(pl.col('n') >= MIN_N_PAIR)
    .select(COL_GENE, COL_TRAIT)
    .sort(COL_GENE, COL_TRAIT)
)

analysis = analysis_variants.join(common_pairs, on=[COL_GENE, COL_TRAIT], how='inner')

print(f'Gene-trait units: {common_pairs.height:,}')
print(f'Analysis rows:    {analysis.height:,}')

## 7. Detectable variance using the per-trait noise

For a gene–trait unit $j$ with $n_j$ variants,

$$
SS_{\mathrm{total},j}=\sum_v(\hat\beta_{v,j}-\bar\beta_j)^2,
\qquad
SS_{\mathrm{noise},j}=\left(1-\frac{1}{n_j}\right)\sum_v \sigma_v^2,
\qquad
SS_{\mathrm{detectable},j}=SS_{\mathrm{total},j}-SS_{\mathrm{noise},j}.
$$

Pooled across units with weight $w_j=n_j-1$, this is the **variance ceiling**: the amount of
between-variant phenotypic variance that is detectable above the sampling-noise floor.

In [ ]:
def build_ceiling_strata(df: pl.DataFrame) -> pl.DataFrame:
    rows = []
    for key, d in df.partition_by([COL_GENE, COL_TRAIT], as_dict=True).items():
        gene, trait = key
        y = d['beta'].to_numpy().astype(float)
        sigma2 = d['sigma2'].to_numpy().astype(float)
        n = len(y)
        yc = y - y.mean()

        SS_total = float(np.dot(yc, yc))
        SS_noise = float((1.0 - 1.0 / n) * sigma2.sum())

        rows.append({
            COL_GENE: gene, COL_TRAIT: trait, 'n': n, 'w': float(n - 1),
            'SS_total': SS_total, 'SS_noise': SS_noise, 'SS_detect': SS_total - SS_noise,
        })
    return pl.DataFrame(rows).sort(COL_GENE, COL_TRAIT)


def pooled_ceiling_summary(strata: pl.DataFrame) -> dict:
    W = float(strata['w'].sum())
    V_total = float(strata['SS_total'].sum()) / W
    V_noise = float(strata['SS_noise'].sum()) / W
    V_detect = float(strata['SS_detect'].sum()) / W
    return {
        'n_pairs': strata.height, 'W': W,
        'V_total': V_total, 'V_noise': V_noise, 'V_detect': V_detect,
        'R2_max': V_detect / V_total if V_total > 0 else np.nan,
    }


ceiling_strata = build_ceiling_strata(analysis)
ceiling_point = pooled_ceiling_summary(ceiling_strata)

pl.DataFrame([ceiling_point])

## 8. Bootstrap the detectable ceiling across gene–trait units

The gene–trait unit is the resampling unit.

In [ ]:
def ceiling_bootstrap(strata: pl.DataFrame, n_boot: int = 2000, seed: int = 1) -> pl.DataFrame:
    rng = np.random.default_rng(seed)
    J = strata.height
    w = strata['w'].to_numpy()
    SS_total = strata['SS_total'].to_numpy()
    SS_noise = strata['SS_noise'].to_numpy()
    SS_detect = strata['SS_detect'].to_numpy()

    rows = []
    for b in range(n_boot):
        idx = rng.integers(0, J, size=J)
        W = w[idx].sum()
        V_total = SS_total[idx].sum() / W
        V_noise = SS_noise[idx].sum() / W
        V_detect = SS_detect[idx].sum() / W
        rows.append({
            'bootstrap': b, 'V_total': V_total, 'V_noise': V_noise,
            'V_detect': V_detect,
            'R2_max': V_detect / V_total if V_total > 0 else np.nan,
        })
    return pl.DataFrame(rows)


def bootstrap_ci(x, alpha=0.05):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    return np.quantile(x, [alpha / 2, 0.5, 1 - alpha / 2])


ceiling_boot = ceiling_bootstrap(ceiling_strata, n_boot=N_BOOT, seed=BOOT_SEED)
detect_ci = bootstrap_ci(ceiling_boot['V_detect'])
r2_ci = bootstrap_ci(ceiling_boot['R2_max'])

print(f"V_detect point: {ceiling_point['V_detect']:.6f}")
print(f'V_detect 95% CI: [{detect_ci[0]:.6f}, {detect_ci[2]:.6f}]')
print(f"R2_max point:  {ceiling_point['R2_max']:.6f}")
print(f'R2_max 95% CI:  [{r2_ci[0]:.6f}, {r2_ci[2]:.6f}]')

## 9. Predictor decomposition with the per-trait noise

For predictor score $S_v$, fit separately within each gene–trait unit:
$\hat\beta_v = \hat b + \hat a S_v + r_v$. After centering the score, $x_v=S_v-\bar S$, the
fitted-score projection has leverage $h_v = x_v^2/\sum_u x_u^2$. Under independent
heteroskedastic noise, the noise the fit spuriously captures is
$SS_{\mathrm{captured,noise}}=\sum_v h_v\sigma_v^2$, so

$$
SS_{\mathrm{captured}} = SS_{\mathrm{captured,raw}} - SS_{\mathrm{captured,noise}},
\qquad
SS_{\mathrm{uncaptured}} = SS_{\mathrm{resid}} - SS_{\mathrm{resid,noise}},
$$

with the exact identity $SS_{\mathrm{detectable}} = SS_{\mathrm{captured}} + SS_{\mathrm{uncaptured}}$.

In [ ]:
def fit_predictor_by_pair(df: pl.DataFrame, score_col: str) -> pl.DataFrame:
    rows = []
    for key, d in df.partition_by([COL_GENE, COL_TRAIT], as_dict=True).items():
        gene, trait = key
        y = d['beta'].to_numpy().astype(float)
        s = d[score_col].to_numpy().astype(float)
        sigma2 = d['sigma2'].to_numpy().astype(float)
        n = len(y)

        yc = y - y.mean()
        x_v = s - s.mean()

        SS_total = float(np.dot(yc, yc))
        SS_noise = float((1.0 - 1.0 / n) * sigma2.sum())
        sum_x2 = float(np.dot(x_v, x_v))

        if sum_x2 <= 0:
            slope = 0.0
            fitted_c = np.zeros_like(yc)
            SS_cap_noise = 0.0
        else:
            slope = float(np.dot(x_v, yc) / sum_x2)
            fitted_c = slope * x_v
            h_v = x_v**2 / sum_x2
            SS_cap_noise = float(np.sum(h_v * sigma2))

        resid = yc - fitted_c
        SS_reg = float(np.dot(fitted_c, fitted_c))
        SS_resid = float(np.dot(resid, resid))

        SS_resid_noise = SS_noise - SS_cap_noise
        SS_detect = SS_total - SS_noise
        SS_captured = SS_reg - SS_cap_noise
        SS_uncaptured = SS_resid - SS_resid_noise

        rows.append({
            COL_GENE: gene, COL_TRAIT: trait, 'n': n, 'w': float(n - 1), 'slope': slope,
            'SS_total': SS_total, 'SS_reg': SS_reg, 'SS_resid': SS_resid,
            'SS_noise': SS_noise, 'SS_cap_noise': SS_cap_noise,
            'SS_resid_noise': SS_resid_noise, 'SS_detect': SS_detect,
            'SS_captured': SS_captured, 'SS_uncaptured': SS_uncaptured,
        })
    return pl.DataFrame(rows).sort(COL_GENE, COL_TRAIT)

### Algebraic sanity check

For every gene-trait unit: $SST=SSR+SSE$ and $SS_{\mathrm{detectable}}=SS_{\mathrm{captured}}+SS_{\mathrm{uncaptured}}$.

In [ ]:
example_fit = fit_predictor_by_pair(analysis, annos[0])

raw_error = example_fit.select(
    (pl.col('SS_total') - pl.col('SS_reg') - pl.col('SS_resid')).abs().max()
).item()
corrected_error = example_fit.select(
    (pl.col('SS_detect') - pl.col('SS_captured') - pl.col('SS_uncaptured')).abs().max()
).item()

print('Max raw OLS identity error:', raw_error)
print('Max corrected identity error:', corrected_error)

## 10. Pool predictor performance

With $W=\sum_j(n_j-1)$,
$V_{\mathrm{captured}}=\sum_j SS_{\mathrm{captured},j}/W$,
$V_{\mathrm{uncaptured}}=\sum_j SS_{\mathrm{uncaptured},j}/W$,
$V_{\mathrm{detectable}}=\sum_j SS_{\mathrm{detectable},j}/W$.
The saturation fraction is $F_{\mathrm{captured}}=V_{\mathrm{captured}}/V_{\mathrm{detectable}}$;
$V_{\mathrm{detectable}}$ is identical for every predictor, since the analysis variants are the
same for all of them (the bootstrap below resamples gene–trait units directly on the corrected
variance components, never on the ratio).

In [ ]:
def pooled_predictor_summary(strata: pl.DataFrame) -> dict:
    W = float(strata['w'].sum())
    V_total = float(strata['SS_total'].sum()) / W
    V_noise = float(strata['SS_noise'].sum()) / W
    V_detect = float(strata['SS_detect'].sum()) / W
    V_captured = float(strata['SS_captured'].sum()) / W
    V_uncaptured = float(strata['SS_uncaptured'].sum()) / W
    return {
        'V_total': V_total, 'V_noise': V_noise, 'V_detect': V_detect,
        'V_captured': V_captured, 'V_uncaptured': V_uncaptured,
        'R2_max': V_detect / V_total if V_total > 0 else np.nan,
        'F_captured': V_captured / V_detect if V_detect != 0 else np.nan,
    }


def predictor_bootstrap(strata: pl.DataFrame, n_boot: int = 2000, seed: int = 1) -> pl.DataFrame:
    rng = np.random.default_rng(seed)
    J = strata.height
    w = strata['w'].to_numpy()
    SS_detect = strata['SS_detect'].to_numpy()
    SS_captured = strata['SS_captured'].to_numpy()
    SS_uncaptured = strata['SS_uncaptured'].to_numpy()

    rows = []
    for b in range(n_boot):
        idx = rng.integers(0, J, size=J)
        W = w[idx].sum()
        V_detect = SS_detect[idx].sum() / W
        V_captured = SS_captured[idx].sum() / W
        V_uncaptured = SS_uncaptured[idx].sum() / W
        rows.append({
            'bootstrap': b, 'V_detect': V_detect, 'V_captured': V_captured,
            'V_uncaptured': V_uncaptured,
            'F_captured': V_captured / V_detect if V_detect != 0 else np.nan,
        })
    return pl.DataFrame(rows)

## 11. Gene–trait-unit bootstrap for every predictor

The bootstrap resamples gene–trait units with replacement. The corrected variance components are never clipped.

In [ ]:
all_results = []

for i, a in enumerate(annos):
    label = anno_to_label.get(a, a)

    fit = fit_predictor_by_pair(analysis, a)
    assert fit.height == common_pairs.height

    point = pooled_predictor_summary(fit)
    assert np.isclose(point['V_detect'], ceiling_point['V_detect'], rtol=1e-10, atol=1e-12)

    boot = predictor_bootstrap(fit, n_boot=N_BOOT, seed=BOOT_SEED + i)
    F_ci = bootstrap_ci(boot['F_captured'])
    Vcap_ci = bootstrap_ci(boot['V_captured'])

    all_results.append({
        'predictor': label, 'n_pairs': common_pairs.height,
        'F_captured': point['F_captured'], 'F_captured_lo': F_ci[0], 'F_captured_hi': F_ci[2],
        'V_captured': point['V_captured'], 'V_captured_lo': Vcap_ci[0], 'V_captured_hi': Vcap_ci[2],
        'V_uncaptured': point['V_uncaptured'], 'V_detect': point['V_detect'],
        'R2_max': point['R2_max'],
    })

all_results = pl.DataFrame(all_results)
all_results

## 12. Sanity check: the ceiling is identical across predictors

The `Vdetect`/`R2max` columns below should be constant across all rows.

In [ ]:
all_results.select('predictor', 'V_detect', 'R2_max')

## 13. Main figure — pooled detectable and captured variance per predictor

Bar plot of pooled variance: the first bar is $V_{\mathrm{detectable}}$ (the common denominator
from Section 7); the remaining bars are each predictor's $V_{\mathrm{captured}}$, ordered
descending, in the same units. Each bar carries a gene–trait-unit bootstrap CI on that quantity
directly (not on the $F_{\mathrm{captured}}$ ratio). A right-hand axis shows the same values as a
percentage of $V_{\mathrm{detectable}}$.

In [ ]:
plot_df = all_results.sort('V_captured', descending=True)

rows = [{
    'label': 'Detectable', 'V': ceiling_point['V_detect'],
    'lo': detect_ci[0], 'hi': detect_ci[2], 'kind': 'Detectable',
}]
for pred, v, lo, hi in zip(
    plot_df['predictor'].to_list(), plot_df['V_captured'].to_list(),
    plot_df['V_captured_lo'].to_list(), plot_df['V_captured_hi'].to_list(),
):
    rows.append({'label': pred, 'V': v, 'lo': lo, 'hi': hi, 'kind': 'Captured'})

bar_df = pd.DataFrame(rows)
bar_df['label'] = pd.Categorical(bar_df['label'], categories=bar_df['label'].tolist(), ordered=True)


def build_bar_plot(df, title):
    return (
        ggplot(df, aes(x='label', y='V', fill='kind'))
        + geom_col()
        + geom_errorbar(aes(ymin='lo', ymax='hi'), width=0.3)
        + geom_hline(yintercept=0, linetype='dotted')
        # + scale_fill_manual(values={'Detectable': '#AFDC2E', 'Captured': '#2A78D6'})
        + scale_fill_manual(values={'Detectable': '#9F72BB', 'Captured': '#3DAED4'})
        + labs(x='', y='Variance', title=title)
        + _THEME
        + theme(
            figure_size=(0.25*len(df) + 1, 5),
            # axis_text_x=element_text(size=12, rotation=45, ha='right'),
            axis_text_x=element_text(size=12, rotation=45, ha='right'),
            axis_text_y=element_text(size=12),
            axis_title=element_text(size=12),
            legend_position='none',
        )
    )


def show_with_pct_axis(p, v_detect):
    '''plotnine has no secondary-axis scale, so this is added directly on the matplotlib
    figure plotnine draws.'''
    fig = p.draw()
    ax = fig.axes[0]
    sec = ax.secondary_yaxis(
        'right', color='gray',
        functions=(lambda v: v / v_detect * 100, lambda pct: pct / 100 * v_detect),
    )
    sec.set_ylabel('% of detectable variance')
    return fig


p = show_with_pct_axis(
    build_bar_plot(bar_df, vc.get('x_label', variant_class)),
    ceiling_point['V_detect'],
)

p.savefig(FIG_DIR / f'F4_variance_ceiling_{variant_class}_trait_noise.svg', dpi=200, bbox_inches='tight')
p

In [ ]:
bar_df